# SAT-TRACK: YOLO26n-OBB training + evaluation

One model, one job: **train YOLO26-OBB on MAR20, then evaluate the best checkpoint.** No YOLO11, no
side-by-side comparisons, nothing to babysit.

Built for cloud boxes (Colab / Kaggle / Lambda / RunPod). Run `Runtime > Run all` and it will:

1. install and sanity-check the environment (GPU, ultralytics version)
2. **fetch the YOLO26 checkpoint itself** - nothing is uploaded, the weights come from the config
3. find the MAR20 dataset (raw VOC-OBB tree *or* an already-converted YOLO OBB folder), auto-extracting a
   `mar20*.zip` if that is all you uploaded
4. convert raw annotations to YOLO OBB format + write the dataset yaml (skipped when already done)
5. train
6. validate the best checkpoint, print overall + per-class metrics
7. plot curves, run a qualitative check, benchmark latency, and bundle everything into a zip

**Cheap pipeline sanity check first** (1 epoch, 320 px, 5% of the data):
```bash
SATTRACK_SMOKE=1 jupyter nbconvert --to notebook --execute yolo26n_obb_train_eval.ipynb \\
    --output smoke_out.ipynb
```

In [ ]:
"""SAT-TRACK configuration - edit this cell, then Run All."""
import os
from pathlib import Path

CFG = {
    "model": "yolo26n-obb.pt",        # yolo26n/s/m/l/x-obb.pt
    "run_name": "mar20_yolo26n_train",
    "epochs": 100,
    "imgsz": 640,
    "batch": 16,                      # 8-16 on a 16 GB GPU, 32+ on 40 GB
    "patience": 20,                   # early-stop patience
    "workers": 8,                     # dataloader workers; 2 on low-RAM boxes
    "device": 0,                      # 0 = first GPU, "cpu" = no GPU
    "seed": 0,
    "fraction": 1.0,                  # 0.05 for a fast pipeline check
    "project": str(Path("runs/obb").resolve()),   # absolute: ultralytics nests relative projects
    "weights_dir": "weights",         # downloaded / uploaded checkpoints land here
    "conf": 0.25,                     # inference confidence for the qualitative check
}

# SATTRACK_SMOKE=1 shrinks everything to a single throwaway epoch
if os.environ.get("SATTRACK_SMOKE") == "1":
    CFG.update(epochs=1, imgsz=320, batch=8, fraction=0.05, workers=2,
               run_name=CFG["run_name"] + "_smoke")

# Places searched for the dataset (first hit wins). Add your own mount path.
DATA_SEARCH_ROOTS = [
    Path.cwd(),
    Path("/content"), Path("/content/drive/MyDrive"),
    Path("/kaggle/input"), Path("/kaggle/working"),
    Path("/workspace"), Path("/data"), Path("/mnt/data"),
    Path.home(),
]
DATA_ROOT = os.environ.get("SATTRACK_DATA")   # optional explicit dataset path

print("config:")
for k, v in CFG.items():
    print(f"  {k:10s} {v}")
print("DATA_ROOT override:", DATA_ROOT)

## 0. Environment check

Installs `ultralytics` only if it is missing, then reports what hardware we actually got. If there is no
CUDA device the notebook falls back to CPU (it works, it is just slow).

In [ ]:
import subprocess
import sys


def ensure_package(spec, module):
    """Import `module`, pip-installing `spec` first if it is absent."""
    try:
        mod = __import__(module)
        print(f"{module} {getattr(mod, '__version__', '?')} already present")
    except ImportError:
        print(f"installing {spec} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-qU", spec], check=True)


ensure_package("ultralytics>=8.4.100", "ultralytics")   # >=8.4 ships yolo26*-obb.pt
ensure_package("pandas", "pandas")
ensure_package("matplotlib", "matplotlib")

import ultralytics
import torch

print(f"\nultralytics: {ultralytics.__version__}")
print(f"torch:       {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU:         {props.name}  ({props.total_memory / 1e9:.1f} GB)")
    CFG["device"] = 0
else:
    print("No CUDA device -> training on CPU.")
    CFG["device"] = "cpu"

## 1. Fetch the YOLO26 checkpoint

No weights get uploaded. This cell resolves `CFG["model"]` to a local `.pt` and downloads it when it is not
on disk. Three accepted forms:

| `CFG["model"]` | behaviour |
|---|---|
| `"yolo26n-obb.pt"` (official asset name) | pulled from the Ultralytics GitHub release into `CFG["weights_dir"]` |
| `"https://.../some.pt"` | downloaded straight from that URL |
| `"weights/best.pt"` (any local file) | used as-is, e.g. your own uploaded or previously trained checkpoint |

Afterwards the cell rewrites `CFG["model"]` to the resolved local path, so training and validation both read
from the file that was actually verified, and prints its parameter count and sha256 so the run is
reproducible.

In [ ]:
import hashlib
from urllib.parse import unquote, urlparse

from ultralytics import YOLO
from ultralytics.utils.downloads import attempt_download_asset, safe_download

WEIGHTS_DIR = Path(CFG["weights_dir"]).expanduser().resolve()
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

src = str(CFG["model"]).strip()
is_url = src.lower().startswith(("http://", "https://"))

if is_url:
    name = unquote(urlparse(src).path).rsplit("/", 1)[-1] or "model.pt"
    MODEL_PATH = WEIGHTS_DIR / name
    if MODEL_PATH.exists():
        print(f"already downloaded: {MODEL_PATH}")
    else:
        print(f"downloading {src}\n  -> {MODEL_PATH}")
        safe_download(url=src, file=str(MODEL_PATH), min_bytes=1e5)
else:
    local = Path(src).expanduser()
    if local.is_file():
        MODEL_PATH = local.resolve()
        print(f"using local checkpoint: {MODEL_PATH}")
    else:
        name = local.name if local.name.endswith(".pt") else f"{local.name}-obb.pt"
        MODEL_PATH = WEIGHTS_DIR / name
        if MODEL_PATH.exists():
            print(f"already downloaded: {MODEL_PATH}")
        else:
            print(f"fetching Ultralytics asset '{name}' -> {MODEL_PATH}")
            attempt_download_asset(str(MODEL_PATH))

if not MODEL_PATH.is_file() or MODEL_PATH.stat().st_size < 1e5:
    raise FileNotFoundError(
        f"checkpoint missing or truncated: {MODEL_PATH}. Check the name in CFG['model'] against "
        "https://github.com/ultralytics/assets/releases (it must be a *-obb.pt file), or pass a direct URL."
    )

MODEL_SHA256 = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()

probe = YOLO(str(MODEL_PATH))
if getattr(probe, "task", None) not in (None, "obb"):
    raise ValueError(f"{MODEL_PATH.name} is a '{probe.task}' model - this notebook needs an OBB checkpoint")

MODEL_PARAMS = sum(p.numel() for p in probe.model.parameters())
print(f"task:   {probe.task}")
print(f"params: {MODEL_PARAMS / 1e6:.2f}M")
print(f"size:   {MODEL_PATH.stat().st_size / 1e6:.1f} MB")
print(f"sha256: {MODEL_SHA256}")

CFG["model"] = str(MODEL_PATH)          # downstream cells train/validate from the resolved file

## 2. Find the dataset

Looks for either layout, anywhere under `DATA_SEARCH_ROOTS`:

* **raw MAR20** - `Annotations/Oriented Bounding Boxes/*.xml`, `JPEGImages/*.jpg`, `ImageSets/Main/train.txt`
* **converted** - `images/{train,val}/*.jpg` + `labels/{train,val}/*.txt`

An already-converted folder always wins (no re-conversion cost). If only a `mar20*.zip` is present it gets
extracted next to the archive first. Once the paths are printed, re-running `Run all` is cheap.

In [ ]:
import zipfile

RAW_MARKERS = (
    Path("Annotations") / "Oriented Bounding Boxes",
    Path("JPEGImages"),
    Path("ImageSets") / "Main",
)
SKIP_DIRS = {"runs", ".git", "node_modules", "__pycache__", ".venv", "venv",
             "site-packages", ".ipynb_checkpoints", "predict"}


def is_raw_mar20(p: Path) -> bool:
    return all((p / m).exists() for m in RAW_MARKERS)


def is_converted(p: Path) -> bool:
    return (p / "images" / "train").is_dir() and any((p / "images" / "train").glob("*.jpg"))


def unzip_mar20_archives(roots):
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for zp in list(root.rglob("*.zip"))[:20]:
            if "mar20" not in zp.name.lower():
                continue
            out = zp.with_suffix("")
            if out.is_dir() and any(out.iterdir()):
                print(f"already extracted: {out}")
                continue
            print(f"extracting {zp} -> {out}")
            with zipfile.ZipFile(zp) as z:
                z.extractall(out)


def scan_for(pred, roots, max_depth=4):
    """Breadth-first search for directories satisfying `pred`."""
    hits = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        queue = [(root, 0)]
        while queue:
            cur, depth = queue.pop(0)
            try:
                if pred(cur):
                    hits.append(cur)
                    continue
                if depth >= max_depth:
                    continue
                queue.extend((c, depth + 1) for c in cur.iterdir()
                             if c.is_dir() and not c.name.startswith(".") and c.name not in SKIP_DIRS)
            except OSError:
                continue
    return hits


RAW_DIR = CONVERTED_DIR = None

if DATA_ROOT:
    p = Path(DATA_ROOT).expanduser().resolve()
    if is_converted(p):
        CONVERTED_DIR = p
    elif is_raw_mar20(p):
        RAW_DIR = p
    else:
        raise FileNotFoundError(f"SATTRACK_DATA={p} is neither raw MAR20 nor a converted YOLO OBB dataset")

if RAW_DIR is None and CONVERTED_DIR is None:
    unzip_mar20_archives(DATA_SEARCH_ROOTS)
    converted = scan_for(is_converted, DATA_SEARCH_ROOTS)
    raw = scan_for(is_raw_mar20, DATA_SEARCH_ROOTS)
    if converted:
        CONVERTED_DIR = converted[0]
    if raw:
        RAW_DIR = raw[0]

print("raw MAR20     :", RAW_DIR)
print("converted OBB :", CONVERTED_DIR)
if RAW_DIR is None and CONVERTED_DIR is None:
    raise FileNotFoundError(
        "No dataset found. Upload the MAR20 tree (or a mar20*.zip, or a converted yolo_obb_dataset) "
        "and set SATTRACK_DATA=/path/to/it if it lives outside the searched roots."
    )

## 3. Convert MAR20 -> YOLO OBB (only if needed)

Same logic as `train_yolo_obb.py`: read the rotated 4-point boxes out of the VOC XML, normalise them, write
`class x1 y1 x2 y2 x3 y3 x4 y4` label files, then drop a dataset yaml in the working directory whose `path`
points at the dataset root (works even when the data lives on a read-only cloud mount).

In [ ]:
import random
import shutil
import xml.etree.ElementTree as ET

CLASS_NAMES = [
    "SU-35", "C-130", "C-17", "C-5", "F-16",
    "TU-160", "E-3", "B-52", "P-3C", "B-1B",
    "E-8", "TU-22", "F-15", "KC-135", "F-22",
    "F-A-18", "TU-95", "KC-10", "SU-34", "SU-24",
]
CLASS_MAP = {f"A{i}": i - 1 for i in range(1, 21)}

DATASET_DIR = Path("yolo_obb_dataset").resolve()
DATA_YAML = Path("mar20_obb.yaml").resolve()


def load_split(path: Path) -> set[str]:
    if not path.exists():
        return set()
    return {line.strip() for line in path.read_text().splitlines() if line.strip()}


def get_image_size(xml_path: Path, img_path: Path) -> tuple[int, int]:
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    w, h = int(size.find("width").text), int(size.find("height").text)
    if w > 0 and h > 0:
        return w, h
    from PIL import Image
    with Image.open(img_path) as im:
        return im.size


def convert_xml_to_obb(xml_path: Path, img_w: int, img_h: int) -> list[str]:
    root = ET.parse(xml_path).getroot()
    lines = []
    for obj in root.findall("object"):
        name = obj.find("name").text
        if name not in CLASS_MAP:
            continue
        rb = obj.find("robndbox")
        x1 = float(rb.find("x_left_top").text) / img_w
        y1 = float(rb.find("y_left_top").text) / img_h
        x2 = float(rb.find("x_right_top").text) / img_w
        y2 = float(rb.find("y_right_top").text) / img_h
        x3 = float(rb.find("x_right_bottom").text) / img_w
        y3 = float(rb.find("y_right_bottom").text) / img_h
        x4 = float(rb.find("x_left_bottom").text) / img_w
        y4 = float(rb.find("y_left_bottom").text) / img_h
        lines.append(f"{CLASS_MAP[name]} {x1:.6f} {y1:.6f} {x2:.6f} {y2:.6f} "
                     f"{x3:.6f} {y3:.6f} {x4:.6f} {y4:.6f}")
    return lines


def build_dataset(raw_root: Path, out_dir: Path = DATASET_DIR) -> None:
    ann_dir = raw_root / "Annotations" / "Oriented Bounding Boxes"
    img_dir = raw_root / "JPEGImages"
    sets_dir = raw_root / "ImageSets" / "Main"

    train_ids, val_ids = load_split(sets_dir / "train.txt"), load_split(sets_dir / "test.txt")
    if not train_ids or not val_ids:                      # no split files: deterministic 90/10
        stems = sorted(p.stem for p in ann_dir.glob("*.xml"))
        random.Random(0).shuffle(stems)
        cut = max(1, int(len(stems) * 0.9))
        train_ids, val_ids = set(stems[:cut]), set(stems[cut:])
        print(f"WARNING: no ImageSets/Main split found - using random 90/10 "
              f"({len(train_ids)}/{len(val_ids)}); this may leak airfields across splits")

    for split, ids in (("train", train_ids), ("val", val_ids)):
        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
        done = 0
        for img_id in sorted(ids):
            xml_path, img_path = ann_dir / f"{img_id}.xml", img_dir / f"{img_id}.jpg"
            if not xml_path.exists() or not img_path.exists():
                continue
            w, h = get_image_size(xml_path, img_path)
            lines = convert_xml_to_obb(xml_path, w, h)
            shutil.copy2(img_path, out_dir / "images" / split / f"{img_id}.jpg")
            (out_dir / "labels" / split / f"{img_id}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
            done += 1
            if done % 250 == 0:
                print(f"  {split}: {done} images converted")
        print(f"{split}: {done} images")


def write_dataset_yaml(dataset_dir: Path, yaml_path: Path = DATA_YAML) -> Path:
    content = (
        "# MAR20 YOLO OBB dataset config (auto-generated by yolo26n_obb_train_eval.ipynb)\n"
        f"path: {Path(dataset_dir).resolve()}\n"
        "train: images/train\n"
        "val: images/val\n\n"
        f"nc: {len(CLASS_NAMES)}\n"
        f"names: {CLASS_NAMES}\n"
    )
    yaml_path.write_text(content)
    return yaml_path


if CONVERTED_DIR is not None:
    DATASET_DIR = Path(CONVERTED_DIR).resolve()
    print(f"using existing converted dataset: {DATASET_DIR} (no conversion needed)")
else:
    print(f"converting raw MAR20 from {RAW_DIR} ...")
    build_dataset(RAW_DIR)

print("dataset yaml ->", write_dataset_yaml(DATASET_DIR))
print("train images:", len(list((DATASET_DIR / 'images' / 'train').glob('*.jpg'))))
print("val images:  ", len(list((DATASET_DIR / 'images' / 'val').glob('*.jpg'))))

## 4. Train

`yolo26n-obb.pt` is downloaded automatically on first use. Weights and the full run (curves, batch mosaics,
`args.yaml`, `results.csv`) land in `runs/obb/<run_name>/`.

In [ ]:
from ultralytics import YOLO

yolo = YOLO(MODEL_PATH)   # resolved (and downloaded) by the fetch cell

yolo.train(
    data=str(DATA_YAML),
    epochs=CFG["epochs"],
    imgsz=CFG["imgsz"],
    batch=CFG["batch"],
    device=CFG["device"],
    patience=CFG["patience"],
    workers=CFG["workers"],
    seed=CFG["seed"],
    fraction=CFG["fraction"],
    project=CFG["project"],
    name=CFG["run_name"],
    exist_ok=True,
    pretrained=True,
    plots=True,
    val=True,
)

RUN_DIR = Path(str(getattr(yolo.trainer, "save_dir", Path(CFG["project"]) / CFG["run_name"])))
BEST = RUN_DIR / "weights" / "best.pt"
print(f"\nrun dir: {RUN_DIR}")
print(f"best weights: {BEST}  (exists: {BEST.exists()})")

## 5. Evaluate the best checkpoint on the held-out split

Full validation pass over the 2,511-image held-out set: overall mAP / precision / recall, then the
per-class table sorted by AP@0.5 (that is where the fighter classes show up as the hard part).

In [ ]:
import numpy as np
import pandas as pd

weights = BEST if BEST.exists() else RUN_DIR / "weights" / "last.pt"
print(f"validating with {weights}")

eval_model = YOLO(str(weights))
metrics = eval_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=CFG["imgsz"],
    batch=CFG["batch"],
    device=CFG["device"],
    plots=True,
    project=CFG["project"],
    name=CFG["run_name"] + "_val",
)
VAL_DIR = Path(str(metrics.save_dir)) if getattr(metrics, "save_dir", None) else RUN_DIR.parent / (CFG["run_name"] + "_val")

box = metrics.box
names = metrics.names if isinstance(metrics.names, dict) else dict(enumerate(metrics.names))
# ap50/ap/p/r only carry the classes actually present in the val split -> map through ap_class_index
cls_idx = np.asarray(getattr(box, "ap_class_index", range(len(names))), dtype=int).ravel()
instances = np.asarray(metrics.nt_per_class).ravel()

overall = pd.DataFrame([{
    "model": CFG["model"],
    "mAP50": box.map50,
    "mAP50-95": box.map,
    "mAP75": box.map75,
    "precision": box.mp,
    "recall": box.mr,
}]).round(4)

per_class = pd.DataFrame({
    "class": [names[int(i)] for i in cls_idx],
    "mAP50": np.asarray(box.ap50),
    "mAP50-95": np.asarray(box.ap),
    "precision": np.asarray(box.p),
    "recall": np.asarray(box.r),
    "instances": instances[cls_idx],
}).round(4).sort_values("mAP50", ascending=False, ignore_index=True)

print("\n=== overall ===")
print(overall.to_string(index=False))
print("\n=== per class (sorted by mAP50) ===")
print(per_class.to_string(index=False))
print(f"\nval plots: {VAL_DIR}")

## 6. Curves and confusion matrix

In [ ]:
from IPython.display import Image, display

def show_if_exists(path, width=760):
    p = Path(path)
    if p.exists():
        print(p)
        display(Image(filename=str(p), width=width))
    else:
        print(f"(missing: {p})")

for candidate in [RUN_DIR / "results.png",
                  VAL_DIR / "confusion_matrix_normalized.png",
                  VAL_DIR / "BoxPR_curve.png",
                  RUN_DIR / "BoxF1_curve.png"]:
    show_if_exists(candidate)

## 7. Qualitative check

A deterministic sample of held-out images, annotated with the model's OBB predictions at `CFG["conf"]`.
Eyeball the fighters here - dense parking aprons are where mistakes hide.

In [ ]:
import matplotlib.pyplot as plt

val_images = sorted((DATASET_DIR / "images" / "val").glob("*.jpg"))
sample = random.Random(0).sample(val_images, min(4, len(val_images)))

fig, axes = plt.subplots(2, 2, figsize=(15, 15))
for ax, img_path in zip(axes.ravel(), sample):
    res = eval_model.predict(str(img_path), imgsz=CFG["imgsz"], device=CFG["device"],
                             conf=CFG["conf"], verbose=False)[0]
    n_det = 0 if res.obb is None else len(res.obb)
    ax.imshow(res.plot()[:, :, ::-1])          # plot() returns BGR
    ax.set_title(f"{img_path.name} - {n_det} detections")
    ax.axis("off")
for ax in axes.ravel()[len(sample):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Latency

Per-image end-to-end predict time (preprocess + inference + post-process). With the NMS-free YOLO26 head the
floor is constant instead of scaling with the number of boxes on screen.

In [ ]:
import time

import numpy as np

bench_imgs = val_images[:25]
for p in bench_imgs[:5]:                       # warm-up (CUDA kernels, cuDNN autotune)
    eval_model.predict(str(p), imgsz=CFG["imgsz"], device=CFG["device"], verbose=False)

latencies = []
for p in bench_imgs:
    t0 = time.perf_counter()
    eval_model.predict(str(p), imgsz=CFG["imgsz"], device=CFG["device"], verbose=False)
    latencies.append((time.perf_counter() - t0) * 1000)

LATENCY_MS = float(np.mean(latencies))
print(f"{len(bench_imgs)} images @ {CFG['imgsz']}px on {CFG['device']}")
print(f"mean {LATENCY_MS:.2f} ms/img | median {np.median(latencies):.2f} | "
      f"p95 {np.percentile(latencies, 95):.2f} | {1000 / LATENCY_MS:.1f} img/s")
print("ultralytics breakdown:", metrics.speed)

## 9. Bundle the artifacts

In [ ]:
import shutil

RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

overall.to_csv(RESULTS / "overall_metrics.csv", index=False)
per_class.to_csv(RESULTS / "per_class_metrics.csv", index=False)
for fname in ["results.png", "confusion_matrix_normalized.png", "BoxPR_curve.png", "BoxF1_curve.png"]:
    src = RUN_DIR / fname
    if src.exists():
        shutil.copy2(src, RESULTS / fname)
if (RUN_DIR / "results.csv").exists():
    shutil.copy2(RUN_DIR / "results.csv", RESULTS / "training_results.csv")

summary_lines = [
    f"# SAT-TRACK - {CFG['model']} on MAR20",
    "",
    f"- run: `{RUN_DIR}`",
    f"- epochs: {CFG['epochs']} (patience {CFG['patience']}), imgsz {CFG['imgsz']}, batch {CFG['batch']}",
    f"- device: {CFG['device']}",
    f"- checkpoint: `{Path(CFG['model']).name}` (sha256 `{MODEL_SHA256}`)",
    f"- latency: {LATENCY_MS:.2f} ms/img",
    "",
    "## Overall",
    "",
    overall.to_string(index=False),
    "",
    "## Per class",
    "",
    per_class.to_string(index=False),
    "",
]
(RESULTS / "summary.md").write_text("\n".join(summary_lines))

bundle = shutil.make_archive("yolo26n_obb_results", "zip", root_dir="results")
print("summary ->", RESULTS / "summary.md")
print("bundle  ->", Path(bundle).resolve())

try:                                            # Colab convenience
    from google.colab import files
    files.download(bundle)
except Exception:
    print("(not Colab - download the zip yourself)")

---

### Notes

* **Bigger variants:** set `CFG["model"]` to `yolo26s-obb.pt` / `yolo26m-obb.pt` / `yolo26l-obb.pt` and bump
  `batch` down a notch if you hit OOM. Everything else stays the same.
* **Re-running just the evaluation:** keep the run directory and start the notebook from section 5
  (`BEST` is resolved from `CFG["project"]` / `CFG["run_name"]`), or point `CFG["model"]` at that
  checkpoint and run only the fetch + evaluate cells.
* **Resume an interrupted run:** `YOLO("runs/obb/<run_name>/weights/last.pt").train(resume=True)`.
* **WandB / cloud logging:** ultralytics picks it up automatically when `WANDB_API_KEY` (or `COMET_*`,
  `MLFLOW_*`) is set in the environment.
* `SATTRACK_SMOKE=1` runs a 1-epoch / 320 px / 5% pass so you can prove the whole pipeline before spending
  GPU hours.